# Day 6: Classify Text Sentiment

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords', quiet=True)

## 1. Load Data

In [ ]:
# Load the Amazon Fine Food Reviews dataset
# Loading a sample of 20,000 rows for fast execution
file_path = "https://media.githubusercontent.com/media/Vvijayaragupathy-uno/machinelearning/main/Day-6/Reviews.csv"
data = pd.read_csv(file_path, nrows=20000)

print(data.head())

## 2. Understanding the Data (EDA)

In [ ]:
# Check how many rows and columns
print("Data Shape:")
print(data.shape)

# Check data types
print("\nData Info:")
print(data.info())

# Check for missing values
print("\nMissing Values:")
print(data.isnull().sum())

# Plot the distribution of review scores (1 to 5 stars)
plt.figure(figsize=(6, 4))
sns.countplot(data=data, x='Score')
plt.title("Distribution of Review Scores (1-5 Stars)")
plt.show()

## 3. Data Cleaning & Sentiment Labeling

In [ ]:
# Filter out neutral reviews (Score == 3) for binary classification
data = data[data['Score'] != 3]

# Create binary sentiment label: 1 for Positive (4-5 stars), 0 for Negative (1-2 stars)
data['Sentiment'] = data['Score'].apply(lambda x: 1 if x > 3 else 0)

# Drop rows with missing text if any
data = data.dropna(subset=['Text'])

print("Sentiment Distribution:")
print(data['Sentiment'].value_counts())

## 4. Text Preprocessing Pipeline

In [ ]:
# Setup Stopwords and Stemmer
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    # 1. Lowercase text
    text = str(text).lower()
    # 2. Remove punctuation and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # 3. Tokenize into words
    words = text.split()
    # 4. Remove stopwords & apply Stemming
    cleaned = [stemmer.stem(word) for word in words if word not in stop_words]
    # 5. Rejoin into string
    return ' '.join(cleaned)

# Apply preprocessing to our reviews
data['Clean_Text'] = data['Text'].apply(preprocess_text)

print("Original Review:")
print(data['Text'].iloc[0])
print("\nCleaned & Preprocessed Review:")
print(data['Clean_Text'].iloc[0])

## 5. Feature Extraction (TF-IDF)

In [ ]:
X = data['Clean_Text']
y = data['Sentiment']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Extract TF-IDF Features
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF Train Matrix Shape:", X_train_tfidf.shape)
print("TF-IDF Test Matrix Shape:", X_test_tfidf.shape)

## 6. Model Training & Evaluation

In [ ]:
# Train Logistic Regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

# Predict on test data
y_pred = model.predict(X_test_tfidf)

# Calculate Accuracy & F1-Score
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy Score:", round(accuracy, 4))
print("F1 Score:", round(f1, 4))

## 7. Top 10 Most Important Words per Class

In [ ]:
# Get vocabulary feature names and model weights
feature_names = np.array(tfidf.get_feature_names_out())
coefficients = model.coef_[0]

# Top 10 words indicating Negative Sentiment (lowest weights)
top_neg_idx = coefficients.argsort()[:10]
top_neg_words = feature_names[top_neg_idx]

# Top 10 words indicating Positive Sentiment (highest weights)
top_pos_idx = coefficients.argsort()[-10:][::-1]
top_pos_words = feature_names[top_pos_idx]

# Summary DataFrame
top_words_df = pd.DataFrame({
    'Top 10 Negative Words': top_neg_words,
    'Top 10 Positive Words': top_pos_words
})

display(top_words_df)